In [ ]:
from pathlib import Path
import gc
import importlib.util
import os
import subprocess
import sys
import time

try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False
if IN_COLAB:
    from google.colab import drive as colab_drive
    colab_drive.mount("/content/drive", force_remount=False)
    WORKSPACE = Path("/content/drive/MyDrive/Zhong et al. 2025 - Neuromatch Team Workspace")
    CODE = WORKSPACE / "code"
    CACHE = Path("/content/zhong-cache")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "scipy>=1.11,<2"], check=True)
    DATABASE = WORKSPACE / "zhong.duckdb"
else:
    WORKSPACE = next(
        path for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "code").is_dir() and (path / "data" / "cache" / "zhong.duckdb").is_file()
    )
    CODE = WORKSPACE / "code"
    CACHE = WORKSPACE / "data" / "cache"
    DATABASE = CACHE / "zhong.duckdb"
os.environ.setdefault("MPLCONFIGDIR", str(WORKSPACE / ".matplotlib"))
if str(CODE) not in sys.path:
    sys.path.insert(0, str(CODE))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import drive

db = drive.setup(cache=str(CACHE), database=str(DATABASE), mount=False)
assert db.database_path.is_file()
db


In [ ]:
from dprime import AREAS, dprime_summary_tables, session_dprime_history

TRIAL_BINS = 5
NEURONS_PER_AREA = 1000
DPRIME_THRESHOLD = 0.3
METRICS = ("mean_abs_dprime", "sd_dprime", "skewness", "excess_kurtosis", "frac_selective")
OUT = drive.results("tanya")
OUT.mkdir(parents=True, exist_ok=True)
{
    "areas": AREAS,
    "trial_bins": TRIAL_BINS,
    "neurons_per_area": NEURONS_PER_AREA,
    "dprime_threshold": DPRIME_THRESHOLD,
}


In [ ]:
manifest = db.query("""
    SELECT b.behavior_session_id, b.behavior_key, b.recording_id,
           b.experiment, b.mouse, b.cohort, e.stage, e.moment, b.trial_count
    FROM behavior_sessions AS b
    JOIN recordings AS r USING (recording_id)
    JOIN experiments AS e USING (experiment)
    WHERE r.has_behavior AND r.has_reduced_neural AND r.has_retinotopy
      AND e.stage = 'train1' AND e.moment IN ('before', 'after')
      AND b.cohort IN ('supervised', 'unsupervised')
    ORDER BY b.cohort, b.mouse, e.moment, b.recording_id
""")
manifest


In [ ]:
assert len(manifest) == 26 and manifest["mouse"].nunique() == 13
db.register("session_manifest", manifest)
db.query("""
    SELECT cohort, moment, COUNT(DISTINCT mouse) AS mice,
           COUNT(*) AS sessions, MIN(trial_count) AS min_trials,
           MAX(trial_count) AS max_trials
    FROM session_manifest
    GROUP BY ALL
    ORDER BY cohort, moment
""")


In [ ]:
probe = next(manifest.itertuples(index=False))
probe_result = session_dprime_history(
    db, probe, trial_bins=TRIAL_BINS, neurons_per_area=NEURONS_PER_AREA,
    threshold=DPRIME_THRESHOLD, verify=True,
)
db.register("probe_history", probe_result)
db.query("""
    SELECT trial_bin, area, pair_count, n_neurons, available_neurons,
           median, sd_dprime, skewness, excess_kurtosis, frac_selective
    FROM probe_history
    ORDER BY trial_bin, area
""")


In [ ]:
scan = [probe_result]
for index, session in enumerate(manifest.itertuples(index=False), start=1):
    if session.behavior_session_id == probe.behavior_session_id:
        continue
    started = time.perf_counter()
    result = session_dprime_history(
        db, session, trial_bins=TRIAL_BINS, neurons_per_area=NEURONS_PER_AREA,
        threshold=DPRIME_THRESHOLD,
    )
    scan.append(result)
    print(f"{index:02d}/26 {session.behavior_session_id} {len(result):,} rows {time.perf_counter() - started:.1f}s")

history = pd.concat(scan, ignore_index=True)
assert len(history) == 26 * len(AREAS) * TRIAL_BINS
history.shape


In [ ]:
db.register("binned_dprime", history)
qc = db.query("""
    SELECT DISTINCT mouse, cohort, moment, trial_count, recorded_neurons
    FROM binned_dprime
    ORDER BY CASE cohort WHEN 'supervised' THEN 0 ELSE 1 END, mouse, moment
""")
db.register("recording_qc", qc)
db.query("""
    SELECT cohort, moment, COUNT(DISTINCT mouse) AS mice,
           ROUND(AVG(trial_count), 1) AS mean_trials,
           ROUND(AVG(recorded_neurons), 1) AS mean_recorded_neurons
    FROM recording_qc
    GROUP BY ALL
    ORDER BY cohort, moment
""")


In [ ]:
qc_wide = db.query("""
    SELECT mouse, cohort,
           MAX(trial_count) FILTER (WHERE moment = 'before') AS trials_before,
           MAX(trial_count) FILTER (WHERE moment = 'after') AS trials_after,
           MAX(recorded_neurons) FILTER (WHERE moment = 'before') AS neurons_before,
           MAX(recorded_neurons) FILTER (WHERE moment = 'after') AS neurons_after
    FROM recording_qc
    GROUP BY ALL
    ORDER BY cohort, mouse
""")
x = np.arange(len(qc_wide))
fig, axes = plt.subplots(2, 1, figsize=(13, 9))
for axis, before, after, title, ylabel in [
    (axes[0], "trials_before", "trials_after", "Trial count by mouse, before vs. after learning", "Trials recorded"),
    (axes[1], "neurons_before", "neurons_after", "Neuron count by mouse, before vs. after learning", "Neurons recorded"),
]:
    axis.bar(x - 0.2, qc_wide[before], 0.4, label="Before learning")
    axis.bar(x + 0.2, qc_wide[after], 0.4, label="After learning", color="green")
    axis.set(title=title, ylabel=ylabel, xticks=x, xticklabels=qc_wide["mouse"])
    axis.tick_params(axis="x", rotation=45)
    axis.legend(frameon=False)
fig.tight_layout()
fig.savefig(OUT / "trial_neuron_qc.png", dpi=180)
plt.show()


In [ ]:
db.query("""
    SELECT area, MIN(available_neurons) AS minimum_available,
           MAX(available_neurons) AS maximum_available,
           MIN(n_neurons) AS minimum_used, MAX(n_neurons) AS maximum_used,
           COUNT(DISTINCT behavior_session_id) AS sessions
    FROM binned_dprime
    GROUP BY area
    ORDER BY area
""")


In [ ]:
trajectory, analysis_windows, sessions, mouse_deltas, exact_tests = dprime_summary_tables(
    history, metrics=METRICS
)
{
    "five-bin_area_rows": history.shape,
    "mouse-first_trajectory_rows": trajectory.shape,
    "session_area_rows": sessions.shape,
    "mouse_delta_rows": mouse_deltas.shape,
    "exact_test_rows": exact_tests.shape,
}


In [ ]:
db.register("dprime_trajectory", trajectory)
db.query("""
    SELECT cohort, moment, progress_bin, mice, mean_abs_dprime, sd_dprime,
           skewness, excess_kurtosis, frac_selective
    FROM dprime_trajectory
    WHERE area = 'mHV'
    ORDER BY cohort, moment, progress_bin
""")


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 7), sharex=True)
for axis, metric in zip(axes.flat, ["sd_dprime", "skewness", "excess_kurtosis", "frac_selective"]):
    current = db.query("""
        SELECT * FROM dprime_trajectory
        WHERE area = 'mHV'
        ORDER BY cohort, moment, progress_bin
    """)
    for (cohort, moment), group in current.groupby(["cohort", "moment"]):
        axis.errorbar(group["progress_bin"], group[metric], yerr=group[f"{metric}_sem"].fillna(0), marker="o", label=f"{cohort} {moment}")
    axis.set(title=f"mHV {metric}", xlabel="trial fraction", xticks=range(1, 6))
axes[0, 0].legend(frameon=False, fontsize=8)
fig.tight_layout()
fig.savefig(OUT / "mhv_distribution_metrics.png", dpi=180)
plt.show()


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 7), sharex=True, sharey=True)
for index, (axis, area) in enumerate(zip(axes.flat, AREAS)):
    current = db.query("""
        SELECT * FROM dprime_trajectory
        WHERE area = ?
        ORDER BY cohort, moment, progress_bin
    """, [area])
    for (cohort, moment), group in current.groupby(["cohort", "moment"]):
        axis.errorbar(group["progress_bin"], group["frac_selective"], yerr=group["frac_selective_sem"].fillna(0), marker="o", label=f"{cohort} {moment}")
    axis.set(title=area, xticks=range(1, 6))
    if index >= 2:
        axis.set_xlabel("trial fraction")
    if index % 2 == 0:
        axis.set_ylabel("fraction |d′| ≥ 0.3")
axes[0, 0].legend(frameon=False, fontsize=8)
fig.tight_layout()
fig.savefig(OUT / "area_selective_fraction.png", dpi=180)
plt.show()


In [ ]:
fig, axis = plt.subplots(figsize=(9, 5))
current = db.query("""
    SELECT * FROM dprime_trajectory
    WHERE area = 'mHV'
    ORDER BY cohort, moment, progress_bin
""")
for (cohort, moment), group in current.groupby(["cohort", "moment"]):
    axis.plot(group["progress_bin"], group["median"], marker="o", label=f"{cohort} {moment}")
    axis.fill_between(group["progress_bin"], group["q05"], group["q95"], alpha=0.1)
axis.axhline(0, color="black", linewidth=0.8)
axis.set(title="mHV d′ distribution, mouse-mean median and 5–95% quantiles", xlabel="trial fraction", ylabel="d′", xticks=range(1, 6))
axis.legend(frameon=False)
fig.tight_layout()
fig.savefig(OUT / "mhv_dprime_quantiles.png", dpi=180)
plt.show()


In [ ]:
db.register("mouse_deltas_view", mouse_deltas)
db.query("""
    SELECT mouse, cohort, metric, before, after, delta
    FROM mouse_deltas_view
    WHERE area = 'mHV'
    ORDER BY metric, cohort, mouse
""")


In [ ]:
db.register("exact_tests_view", exact_tests)
db.query("""
    SELECT area, metric, test, cohort, effect, pvalue, permutations, mice
    FROM exact_tests_view
    WHERE area = 'mHV'
    ORDER BY metric, test, cohort
""")


In [ ]:
outputs = {
    "five_bin_dprime.csv": history,
    "trial_neuron_qc.csv": qc,
    "mouse_first_trajectory.csv": trajectory,
    "session_area_metrics.csv": sessions,
    "mouse_deltas.csv": mouse_deltas,
    "exact_mouse_tests.csv": exact_tests,
}
for filename, frame in outputs.items():
    frame.to_csv(OUT / filename, index=False)
pd.DataFrame({"file": list(outputs), "rows": [len(frame) for frame in outputs.values()]})


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Creates 4 animated GIFs, in this order:
1. Supervised, Before Learning
2. Unsupervised, Before Learning
3. Supervised, After Learning
4. Unsupervised, After Learning

Each shows the pooled d' distribution across all mice in that
group/condition, for the first 100 windows (padded with NaN if a
mouse's session is shorter than 100 windows).
"""

from pathlib import Path
import os
import gc
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from scipy.stats import gaussian_kde

# ------------------------------------------------------------
# Your data folder
# ------------------------------------------------------------
root = Path(os.environ.get("ZHONG_WORKSPACE", Path.cwd()))

N_WINDOWS_TARGET = 100

supervised_mice_bef = {
    'VR2': 'VR2_2021_03_20_1', 'TX60': 'TX60_2021_04_10_1',
    'TX108': 'TX108_2023_03_13_1', 'TX109': 'TX109_2023_03_27_1',
}
supervised_mice_aft = {
    'VR2': 'VR2_2021_04_06_1', 'TX60': 'TX60_2021_05_04_1',
    'TX108': 'TX108_2023_03_22_1', 'TX109': 'TX109_2023_04_07_1',
}
unsupervised_mice_bef = {
    'TX83': 'TX83_2022_08_17_1', 'TX88': 'TX88_2022_06_13_2',
    'TX105': 'TX105_2022_10_08_2', 'TX104': 'TX104_2022_10_12_1',
    'TX119': 'TX119_2023_12_14_1', 'TX123': 'TX123_2023_12_21_1',
    'DR10': 'DR10_2022_07_12_1', 'DR15': 'DR15_2022_10_09_1',
    'TX85': 'TX85_2022_06_14_1',
}
unsupervised_mice_aft = {
    'TX83': 'TX83_2022_08_29_1', 'TX88': 'TX88_2022_06_17_2',
    'TX105': 'TX105_2022_10_19_2', 'TX104': 'TX104_2022_10_20_1',
    'TX119': 'TX119_2023_12_23_1', 'TX123': 'TX123_2024_01_02_1',
    'DR10': 'DR10_2022_07_19_1', 'DR15': 'DR15_2022_10_19_1',
    'TX85': 'TX85_2022_06_17_1',
}

# order: both "before" conditions first, then both "after" conditions
CONDITIONS = [
    ('sup_bef',   supervised_mice_bef,   'Beh_sup_train1_before_learning.npy',   'Supervised, Before Learning'),
    ('unsup_bef', unsupervised_mice_bef, 'Beh_unsup_train1_before_learning.npy', 'Unsupervised, Before Learning'),
    ('sup_aft',   supervised_mice_aft,   'Beh_sup_train1_after_learning.npy',    'Supervised, After Learning'),
    ('unsup_aft', unsupervised_mice_aft, 'Beh_unsup_train1_after_learning.npy',  'Unsupervised, After Learning'),
]


# ------------------------------------------------------------
# Rui's original sliding-window d' function (unchanged)
# ------------------------------------------------------------
def calculate_single_pair_dprime_history(proj_data, stim_fr, pos_fr, VR_move, name_stim1, name_stim2, window_size=20):
    n_neurons, n_frames = proj_data.shape

    stim_fr = stim_fr[:n_frames]
    pos_fr = pos_fr[:n_frames]
    VR_move = VR_move[:n_frames]

    pos_diff = np.diff(pos_fr, prepend=pos_fr[0])
    stim_fr_changed = (stim_fr != np.roll(stim_fr, 1))
    stim_fr_changed[0] = False
    trial_starts = (pos_diff < -10) | stim_fr_changed
    trial_indices = np.cumsum(trial_starts)

    unique_trials = np.unique(trial_indices)

    stim1_trials_frames = []
    stim2_trials_frames = []

    for t in unique_trials:
        t_mask = (trial_indices == t) & (pos_fr < 40) & VR_move
        if np.sum(t_mask) < 3:
            continue

        t_stim_name = stim_fr[t_mask][0]
        trial_data = proj_data[:, t_mask]

        if t_stim_name == name_stim1:
            stim1_trials_frames.append(trial_data)
        elif t_stim_name == name_stim2:
            stim2_trials_frames.append(trial_data)

    n_pairs = min(len(stim1_trials_frames), len(stim2_trials_frames))
    print(f"    Stim1 has {len(stim1_trials_frames)} trials, Stim2 has {len(stim2_trials_frames)} trials, {n_pairs} pairs")

    if n_pairs < window_size:
        print(f"    SKIPPING: not enough pairs ({n_pairs}) for window size ({window_size})")
        return None

    dprime_history = []
    n_windows = n_pairs - window_size + 1

    for w in range(n_windows):
        window_leaf_data = np.hstack(stim1_trials_frames[w: w + window_size])
        window_circle_data = np.hstack(stim2_trials_frames[w: w + window_size])

        u_leaf = np.nanmean(window_leaf_data, axis=1)
        u_circle = np.nanmean(window_circle_data, axis=1)
        sig_leaf = np.nanstd(window_leaf_data, axis=1)
        sig_circle = np.nanstd(window_circle_data, axis=1)

        denominator = sig_leaf + sig_circle + 1e-8
        dp_w = 2 * (u_leaf - u_circle) / denominator
        dprime_history.append(dp_w)

    dprime_history = np.column_stack(dprime_history)
    dprime_history = np.nan_to_num(dprime_history, nan=0.0, posinf=0.0, neginf=0.0)

    return dprime_history


def load_and_compute(session_id, beh_filename, window_size=20):
    beh = np.load(os.path.join(root, beh_filename), allow_pickle=1).item()[session_id]

    stim_id = beh['stim_id']
    uniqW, WallN = beh['UniqWalls'], beh['WallName']
    stim_fr = beh['ft_WallID']
    pos_fr = beh['ft_Pos']
    VR_move = beh['ft_move'] > 0

    name_stim1 = uniqW[stim_id == 2][0]
    name_stim2 = uniqW[stim_id == 0][0]

    svd_dec = np.load(os.path.join(root, session_id + '_SVD_dec.npy'), allow_pickle=1).item()
    # float32 keeps memory use as low as possible
    proj_data = (svd_dec['U'].T @ svd_dec['V']).astype(np.float32)

    # free the raw SVD components immediately, before the d' calculation runs
    del svd_dec
    gc.collect()

    dprime_history = calculate_single_pair_dprime_history(
        proj_data, stim_fr, pos_fr, VR_move, name_stim1, name_stim2, window_size=window_size
    )

    del proj_data, beh
    gc.collect()

    return dprime_history


def pad_to_target(dprime_history, target_windows):
    n_neurons, n_windows = dprime_history.shape
    if n_windows >= target_windows:
        return dprime_history[:, :target_windows]
    else:
        pad_width = target_windows - n_windows
        nan_pad = np.full((n_neurons, pad_width), np.nan)
        return np.hstack([dprime_history, nan_pad])


def process_group(mice_dict, beh_filename, group_name):
    print(f"\n{'=' * 60}")
    print(f"Processing group: {group_name}")
    print('=' * 60)

    all_mice_padded = []

    for mouse_name, session_id in mice_dict.items():
        print(f"\n  Mouse: {mouse_name} ({session_id})")
        try:
            dprime_history = load_and_compute(session_id, beh_filename, window_size=20)
            if dprime_history is None:
                print(f"    Skipping {mouse_name} - not enough data")
                continue

            padded = pad_to_target(dprime_history, N_WINDOWS_TARGET)
            all_mice_padded.append(padded)
            print(f"    {mouse_name}: had {dprime_history.shape[1]} windows, padded/trimmed to {N_WINDOWS_TARGET}")

        except Exception as error:
            print(f"    SKIPPED {mouse_name}: {type(error).__name__}: {error}")
            continue

    pooled_matrix = np.vstack(all_mice_padded)
    print(f"\n  Pooled matrix shape for {group_name}: {pooled_matrix.shape}  (neurons x windows)")

    return pooled_matrix


# ------------------------------------------------------------
# Animation function - fixed x-axis (-1.5 to 1.5), stable y-axis,
# no "(n=...)" in title
# ------------------------------------------------------------
def animate_dprime_distribution(dprime_matrix, fps=10, save_path="dprime_evolution.gif", title_prefix=""):
    n_neurons, n_windows = dprime_matrix.shape
    print(f"Preparing animation: {n_neurons} pooled neurons, {n_windows} windows...")

    # --- X-AXIS: fixed at -1.5 to 1.5 ---
    x_min, x_max = -1.5, 1.5
    x_eval = np.linspace(x_min, x_max, 300)

    # --- Y-AXIS: precompute tallest bar across all frames with real data ---
    print("  Calculating stable y-axis limit across all frames...")
    max_density = 0
    for frame in range(n_windows):
        current_data = dprime_matrix[:, frame]
        current_data = current_data[~np.isnan(current_data)]
        if current_data.size == 0:
            print(f"    Window {frame + 1}: NO real data (0 neurons)")
            continue
        counts, _ = np.histogram(current_data, bins=60, range=(x_min, x_max), density=True)
        max_density = max(max_density, counts.max())
    y_max = max_density * 1.15

    fig, ax = plt.subplots(figsize=(8, 5), dpi=150)

    def update(frame):
        ax.clear()
        current_data = dprime_matrix[:, frame]
        current_data = current_data[~np.isnan(current_data)]

        if current_data.size == 0:
            ax.set_xlim(x_min, x_max)
            ax.set_ylim(0, y_max)
            ax.set_title(f"{title_prefix} | Window {frame + 1}/{n_windows} (no data)", fontsize=12)
            return

        ax.hist(current_data, bins=60, range=(x_min, x_max),
                density=True, alpha=0.4, color='#4A90E2', label='Neurons Hist')

        if len(current_data) > 1000:
            sample_data = np.random.choice(current_data, size=1000, replace=False)
        else:
            sample_data = current_data

        try:
            kde = gaussian_kde(sample_data)
            ax.plot(x_eval, kde(x_eval), color='#D0021B', lw=2.5, label='KDE Fit')
        except Exception:
            pass

        ax.axvline(x=0, color='gray', linestyle='--', alpha=0.7)
        ax.set_xlim(x_min, x_max)
        ax.set_ylim(0, y_max)
        ax.set_xlabel("d-prime value", fontsize=11)
        ax.set_ylabel("Density", fontsize=11)
        ax.set_title(f"{title_prefix} | Window {frame + 1}/{n_windows}", fontsize=12, fontweight='bold')
        ax.legend(loc='upper right')
        ax.grid(axis='y', alpha=0.3)

    anim = FuncAnimation(fig, update, frames=n_windows, interval=1000 / fps)

    print(f"Saving {save_path}, please wait...")
    writer = PillowWriter(fps=fps)
    anim.save(save_path, writer=writer)
    plt.close()
    print("Finished!")


# ------------------------------------------------------------
# Run all 4 conditions, in order: sup_bef, unsup_bef, sup_aft, unsup_aft
# ------------------------------------------------------------
for condition_key, mice_dict, beh_filename, title in CONDITIONS:
    pooled = process_group(mice_dict, beh_filename, title)
    animate_dprime_distribution(
        pooled, fps=12,
        save_path=str(root / f"dprime_GROUP_{condition_key}_first100windows.gif"),
        title_prefix=title
    )

print("\n\nAll 4 group-pooled animations complete!")